# SEIRS-SEI Model with Environmental Modifications

This notebook demonstrates the SEIRS-SEI (Susceptible-Exposed-Infectious-Recovered for Humans, Susceptible-Exposed-Infectious for Mosquitoes) model with environmental modifications.

## Features

- **Human compartments**: S_H, E_H, I_H, R_H (with waning immunity)
- **Mosquito compartments**: S_M, E_M, I_M
- **Environmental drivers**: Temperature, precipitation, humidity
- **Deforestation effects**: Increases breeding sites and human-vector contact
- **Fire effects**: Smoke reduces biting rate, habitat destruction affects recruitment

## Data Sources

- Climate data from API
- Deforestation data from DETER (INPE)
- Fire counts from INPE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from epimodels.continuous import SEIRS_SEI

## Load Environmental Data

In [ ]:
DATA_PATH = './data/'

# Study period
START_DATE = '2017-01-01'
END_DATE = '2023-12-31'

# Load climate data
climate_df = pd.read_csv(f'{DATA_PATH}climate_api_data_2016_2024.csv')
climate_df['date'] = pd.to_datetime(climate_df['date'])

# Load cases data
cases_df = pd.read_csv(f'{DATA_PATH}cumulative_manaus_cases_2016_2023.csv')
cases_df['date'] = pd.to_datetime(cases_df['date'])

# Load fire data
fire_df = pd.read_csv(f'{DATA_PATH}inpe_fire_counts_data_2016_2024.csv')
fire_df['date'] = pd.to_datetime(fire_df['date'])

# Load deforestation data
defor_df = pd.read_csv(f'{DATA_PATH}treated_deter_deforestation_data_2016_2024.csv')
defor_df['date'] = pd.to_datetime(defor_df['date'])

# Filter to study period
climate_df = climate_df[
    (climate_df['date'] >= START_DATE) & 
    (climate_df['date'] <= END_DATE)
].reset_index(drop=True)

cases_df = cases_df[
    (cases_df['date'] >= START_DATE) & 
    (cases_df['date'] <= END_DATE)
].reset_index(drop=True)

fire_df = fire_df[
    (fire_df['date'] >= START_DATE) & 
    (fire_df['date'] <= END_DATE)
].reset_index(drop=True)

defor_df = defor_df[
    (defor_df['date'] >= START_DATE) & 
    (defor_df['date'] <= END_DATE)
].reset_index(drop=True)

# Merge all data
climate_df = climate_df.merge(fire_df[['date', 'fire_counts']], on='date', how='left')
climate_df = climate_df.merge(defor_df[['date', 'total_degradation']], on='date', how='left')
climate_df['fire_counts'] = climate_df['fire_counts'].fillna(0)
climate_df['total_degradation'] = climate_df['total_degradation'].fillna(0)

print(f"Data shape: {climate_df.shape}")
print(f"Date range: {climate_df['date'].min()} to {climate_df['date'].max()}")
print(f"Cases data shape: {cases_df.shape}")
print(f"\nColumns in climate data: {list(climate_df.columns)}")
print(f"Columns in cases data: {list(cases_df.columns)}")

## Create Interpolation Functions

In [ ]:
# Create day number from start
climate_df['day_number'] = (climate_df['date'] - climate_df['date'].min()).dt.days

# Create interpolation functions
temp_interp = interp1d(
    climate_df['day_number'], 
    climate_df['temp_med'],
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate'
)

precip_interp = interp1d(
    climate_df['day_number'],
    climate_df['precip_med'],
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate'
)

umid_interp = interp1d(
    climate_df['day_number'],
    climate_df['umid_med'],
    kind='linear',
    bounds_error=False,
    fill_value='extrapolate'
)

fire_interp = interp1d(
    climate_df['day_number'],
    climate_df['fire_counts'],
    kind='linear',
    bounds_error=False,
    fill_value=0
)

defor_interp = interp1d(
    climate_df['day_number'],
    climate_df['total_degradation'],
    kind='linear',
    bounds_error=False,
    fill_value=0
)

print("Interpolation functions created.")
print(f"Temperature range: {climate_df['temp_med'].min():.1f}°C to {climate_df['temp_med'].max():.1f}°C")
print(f"Precipitation range: {climate_df['precip_med'].min():.1f}mm to {climate_df['precip_med'].max():.1f}mm")
print(f"Humidity range: {climate_df['umid_med'].min():.1f}% to {climate_df['umid_med'].max():.1f}%")
print(f"Fire counts range: {climate_df['fire_counts'].min():.0f} to {climate_df['fire_counts'].max():.0f}")
print(f"Deforestation range: {climate_df['total_degradation'].min():.2f} to {climate_df['total_degradation'].max():.2f}")

## Initialize Model

In [ ]:
# Create model with environmental data functions
model = SEIRS_SEI(
    temp_func=temp_interp,
    precip_func=precip_interp,
    umid_func=umid_interp,
    fire_func=fire_interp,
    defor_func=defor_interp
)

print(f"Model type: {model.model_type}")
print(f"State variables: {list(model.state_variables.keys())}")
print(f"\nModel diagram:")
print(model.diagram)

## Define Parameters

In [ ]:
# Population parameters
N = 8558  # Human population
M = 100000  # Mosquito population

# Fixed model parameters
params = {
    # Transmission probabilities
    'b1': 0.9,  # mosquito -> human
    'b2': 0.9,  # human -> mosquito
    
    # Human disease dynamics
    'gamma': 1/60,  # recovery rate (~60 days)
    'r_H': 0.00005,  # net human growth rate
    'omega': 1/365,  # immunity loss rate (~1 year)
    'tau_H': 10,  # human incubation period (days)
    
    # Mosquito life cycle
    'BE': 200,
    'pME': 0.9,
    'pML': 0.25,
    'pMP': 0.75,
    'tauE': 1,
    'tauP': 1,
    'c1': 0.00554,
    'c2': -0.06737,
    'D1': 4.0,
    'A': 12.5,
    'B': 15.0,
    'C': -48.78,
    
    # Sporogonic cycle
    'DD': 105,  # degree-days
    'Tmin': 14.5,
    'T_prime': 25.6,
    
    # Larval habitat
    'R_L': 32.67,  # rainfall threshold for breeding
    
    # Environmental modification parameters
    'defor_max_effect': 0.3,  # max 30% increase from deforestation
    'defor_scale': 0.0001,
    'defor_delay': 14,  # days delay for deforestation effect
    'fire_smoke_effect': 0.4,  # max 40% reduction from smoke
    'fire_habitat_effect': 0.2,  # 20% reduction from habitat destruction
    'fire_recovery_delay': 21,  # days for post-fire recovery
    
    # Population sizes
    'N': N,
    'M': M
}

print("Parameters defined.")

## Set Initial Conditions

In [ ]:
# Get I_H0 from cases data at the start date
start_date_obj = pd.to_datetime(START_DATE)

# Use the same elif logic from the reference notebook
if 'active_symptomatic' in cases_df.columns and 'active_asymptomatic' in cases_df.columns:
    # Calculate proportion asymptomatic for model parameter
    cases_df['prop_asymptomatic'] = cases_df['active_asymptomatic'] / cases_df['active_total']
    avg_prop_asymp = cases_df['prop_asymptomatic'].mean()
    print(f"Average proportion asymptomatic: {avg_prop_asymp:.2f}")
    
    # Use total_infectious as I_H
    I_H_data = cases_df['active_total'].values
    I_H0 = I_H_data[0]  # Initial infectious at start date
    
    # Estimate E_H0 from I_H0 and incubation period
    # E_H0 ~ half as many exposed as infectious (rough estimate)
    E_H0 = I_H0 * 0.5
    
elif 'cumulative_cases' in cases_df.columns:
    # If you only have cumulative cases, calculate new cases per day
    cases_df['new_cases'] = cases_df['cumulative_cases'].diff().fillna(0)
    I_H_data = cases_df['new_cases'].values
    I_H0 = I_H_data[0]
    
    # Estimate E_H0 from I_H0 and incubation period
    # New cases today ~= were exposed tau_H days ago
    E_H0 = I_H0 * params['tau_H'] / (1/params['gamma'])
    
else:
    # Fallback: if you only have active cases
    I_H_data = cases_df['active_total'].values
    I_H0 = I_H_data[0]
    E_H0 = I_H0 * 0.5  # Estimate

R_H0 = 0  # No recovered initially
S_H0 = N - E_H0 - I_H0 - R_H0

# Mosquito initial conditions
E_M0 = 10000
I_M0 = 500
S_M0 = M - E_M0 - I_M0

# Initial state vector: [S_H, E_H, I_H, R_H, S_M, E_M, I_M]
inits = [S_H0, E_H0, I_H0, R_H0, S_M0, E_M0, I_M0]

print(f"Initial conditions:")
print(f"  S_H0: {S_H0:.0f}")
print(f"  E_H0: {E_H0:.0f}")
print(f"  I_H0: {I_H0:.0f}")
print(f"  R_H0: {R_H0:.0f}")
print(f"  S_M0: {S_M0:.0f}")
print(f"  E_M0: {E_M0:.0f}")
print(f"  I_M0: {I_M0:.0f}")

## Run Model

In [ ]:
# Time range (days)
t_start = climate_df['day_number'].min()
t_end = climate_df['day_number'].max()

print(f"Running model from {START_DATE} to {END_DATE}")
print(f"Day range: {t_start} to {t_end}")

# Run model
model(
    inits=inits,
    trange=[t_start, t_end],
    totpop=N + M,
    params=params,
    validate=False
)

print(f"Model run complete.")
print(f"Solution shape: {len(model.traces['time'])} time points")

## Plot Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Human populations
t = model.traces['time']
axes[0, 0].plot(t, model.traces['S_H'], label='Susceptible (S_H)', linewidth=1.5)
axes[0, 0].plot(t, model.traces['E_H'], label='Exposed (E_H)', linewidth=1.5)
axes[0, 0].plot(t, model.traces['I_H'], label='Infectious (I_H)', linewidth=2)
axes[0, 0].plot(t, model.traces['R_H'], label='Recovered (R_H)', linewidth=1.5)
axes[0, 0].set_xlabel('Time (days)')
axes[0, 0].set_ylabel('Human Population')
axes[0, 0].set_title('Human SEIRS Dynamics')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Mosquito populations (normalized)
axes[0, 1].plot(t, model.traces['S_M'], label='Susceptible (S_M)', linewidth=1.5)
axes[0, 1].plot(t, model.traces['E_M'], label='Exposed (E_M)', linewidth=1.5)
axes[0, 1].plot(t, model.traces['I_M'], label='Infectious (I_M)', linewidth=2)
axes[0, 1].set_xlabel('Time (days)')
axes[0, 1].set_ylabel('Mosquito Population')
axes[0, 1].set_title('Mosquito SEI Dynamics')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Infectious humans vs temperature
ax2 = axes[1, 0]
ax2.plot(t, model.traces['I_H'], 'r-', linewidth=2, label='Infectious Humans')
ax2.set_xlabel('Time (days)')
ax2.set_ylabel('Infectious Humans', color='r')
ax2.tick_params(axis='y', labelcolor='r')

ax2_twin = ax2.twinx()
temp_vals = [float(temp_interp(ti)) for ti in t]
ax2_twin.plot(t, temp_vals, 'orange', alpha=0.7, linewidth=1, label='Temperature')
ax2_twin.set_ylabel('Temperature (°C)', color='orange')
ax2_twin.tick_params(axis='y', labelcolor='orange')
axes[1, 0].set_title('Infectious Humans vs Temperature')

# Fire counts effect
ax3 = axes[1, 1]
ax3.bar(t[::30], [float(fire_interp(ti)) for ti in t[::30]], alpha=0.5, label='Fire Counts', width=25)
ax3.set_xlabel('Time (days)')
ax3.set_ylabel('Fire Counts', color='red')
ax3.tick_params(axis='y', labelcolor='red')

ax3_twin = ax3.twinx()
ax3_twin.plot(t, model.traces['I_H'], 'b-', linewidth=2, label='Infectious Humans')
ax3_twin.set_ylabel('Infectious Humans', color='blue')
ax3_twin.tick_params(axis='y', labelcolor='blue')
axes[1, 1].set_title('Fire Activity and Infection Dynamics')

plt.tight_layout()
plt.show()

## Model Validation: Comparison with Observed Data

In [ ]:
# Prepare model output for comparison
model_years = 2017 + model.traces['time'] / 365.25
model_I_H = model.traces['I_H']

# Prepare observed data for plotting
# Convert dates to continuous years for proper alignment
obs_years = cases_df['date'].dt.year.values + (cases_df['date'].dt.dayofyear.values - 1) / 365.25
obs_active = cases_df['active_total'].values

# Interpolate model output to observed time points
from scipy.interpolate import interp1d
model_interp = interp1d(model_years, model_I_H, kind='linear', 
                         bounds_error=False, fill_value='extrapolate')
model_I_H_at_obs = model_interp(obs_years)

print(f"Model I_H range: {model_I_H.min():.0f} to {model_I_H.max():.0f}")
print(f"Observed active cases range: {obs_active.min():.0f} to {obs_active.max():.0f}")

In [ ]:
# ============================================================================
# PLOT 1: DIRECT COMPARISON - Model vs Observed Data
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Model output (I_H - infectious humans)
ax.plot(model_years, model_I_H, 'b-', linewidth=2, 
        label='Model I_H (Infectious Humans)', alpha=0.8)

# Observed active cases
ax.plot(obs_years, obs_active, 'r-', linewidth=1.5, 
        label='Observed Active Cases (Surveillance Data)', alpha=0.7, 
        marker='o', markersize=3, markevery=30)

# Formatting
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Active Cases / Infectious Individuals', fontsize=12)
ax.set_title('SEIRS-SEI Model Validation: Predicted vs Observed Malaria Cases\n' +
             f'Manaus, Brazil ({START_DATE} to {END_DATE})', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

# Add vertical lines for year boundaries
for year in range(2017, 2025):
    ax.axvline(x=year, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# PLOT 2: NORMALIZED COMPARISON - Pattern Matching
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Normalize both time series to [0, 1] range for pattern comparison
# Normalize model output
if model_I_H.max() > model_I_H.min():
    I_H_norm = (model_I_H - model_I_H.min()) / (model_I_H.max() - model_I_H.min())
else:
    I_H_norm = model_I_H

# Normalize observed data
if obs_active.max() > obs_active.min():
    obs_norm = (obs_active - obs_active.min()) / (obs_active.max() - obs_active.min())
else:
    obs_norm = obs_active

# Plot normalized time series
ax.plot(model_years, I_H_norm, 'b-', linewidth=2, 
        label='Model I_H (normalized)', alpha=0.8)
ax.plot(obs_years, obs_norm, 'r-', linewidth=1.5, 
        label='Observed Cases (normalized)', alpha=0.7, 
        marker='o', markersize=3, markevery=30)

# Formatting
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Normalized Value (0 to 1 scale)', fontsize=12)
ax.set_title('Temporal Pattern Comparison: Model vs Observations\n' +
             '(Normalized to show timing of peaks and troughs)', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

# Add vertical lines for year boundaries
for year in range(2017, 2025):
    ax.axvline(x=year, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Calculate Statistics
# ============================================================================

from scipy.stats import pearsonr, spearmanr

# Calculate correlation coefficients
pearson_r, pearson_p = pearsonr(model_I_H_at_obs, obs_active)
spearman_r, spearman_p = spearmanr(model_I_H_at_obs, obs_active)

# Calculate RMSE
rmse = np.sqrt(np.mean((model_I_H_at_obs - obs_active)**2))

# Calculate mean absolute error
mae = np.mean(np.abs(model_I_H_at_obs - obs_active))

print("="*60)
print("MODEL VALIDATION STATISTICS")
print("="*60)
print(f"Pearson correlation: {pearson_r:.4f} (p-value: {pearson_p:.2e})")
print(f"Spearman correlation: {spearman_r:.4f} (p-value: {spearman_p:.2e})")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print("="*60)

## Time-Dependent R0

In [ ]:
# Calculate R0 over time
model.param_values = params
t_sample = np.linspace(t_start, t_end, 500)
r0_values = [model.R0_t(ti) for ti in t_sample]
r0_values = [r0 if r0 is not None else 0 for r0 in r0_values]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_sample, r0_values, 'b-', alpha=0.7, linewidth=2)
ax.axhline(y=1, color='r', linestyle='--', label='R0 = 1')
ax.set_xlabel('Time (days)')
ax.set_ylabel('R0')
ax.set_title('Time-Dependent Basic Reproduction Number (R0)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"R0 range: {min(r0_values):.2f} to {max(r0_values):.2f}")

## Summary

The SEIRS-SEI model extends the basic vector-borne disease model with:

1. **Human exposed compartment (E_H)**: Models the incubation period in humans
2. **Waning immunity (omega)**: Recovered individuals can become susceptible again
3. **Environmental modifications**:
   - Deforestation increases breeding sites and human exposure
   - Forest fires reduce biting rate (smoke effect) and destroy habitats
4. **Climate-driven dynamics**: Temperature, humidity, and rainfall affect mosquito survival and development